In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score

# Result files in outputs/results/ with their paper labels (Table 1 order) and plot styling.
# Off-the-shelf = light hue + hollow marker; SFT = same hue saturated + filled marker.
MODELS = {
    "statistical-baseline.json":  {"label": "Statistical Baseline",  "color": "#6e7275", "marker": "s", "ls": (0, (5, 2)), "fill": True},
    "ministral-3-3b.json":        {"label": "Ministral 3 3B",        "color": "#74b9ff", "marker": "o", "ls": "-", "fill": False},
    "ministral-3-3b-sft.json":    {"label": "Ministral 3 3B (SFT)",  "color": "#0984e3", "marker": "o", "ls": "-", "fill": True},
    "ministral-3-8b.json":        {"label": "Ministral 3 8B",        "color": "#ff7675", "marker": "^", "ls": "-", "fill": False},
    "ministral-3-8b-sft.json":    {"label": "Ministral 3 8B (SFT)",  "color": "#d63031", "marker": "^", "ls": "-", "fill": True},
    "gemini-3.5-flash-lite.json": {"label": "Gemini 3.5 Flash Lite", "color": "#fdcb6e", "marker": "D", "ls": "-", "fill": True},  # accuracy only (no logits)
}


In [ ]:
def calculate_calibration_data(results: list, num_bins: int = 10):
    """Extracts binned confidences, accuracies, the ECE, and bin distributions, normalized for OOD mass."""
    
    confidences = []
    for r in results:
        p0 = r["probabilities"]["0"]
        p1 = r["probabilities"]["1"]
        total_valid_mass = p0 + p1
        
        # Renormalize to isolate epistemic confidence (ignoring formatting noise)
        if total_valid_mass > 0:
            norm_p0 = p0 / total_valid_mass
            norm_p1 = p1 / total_valid_mass
            confidences.append(max(norm_p0, norm_p1))
        else:
            # Fallback to random chance if the model output 100% OOD tokens
            confidences.append(0.5) 
            
    confidences = np.array(confidences)
    accuracies = np.array([1 if str(r["true_label"]) == str(r["prediction"]) else 0 for r in results])

    bins = np.linspace(0.5, 1.0, num_bins + 1)
    bin_indices = np.digitize(confidences, bins) - 1
    
    bin_accuracies = np.zeros(num_bins)
    bin_confidences = np.zeros(num_bins)
    bin_counts = np.zeros(num_bins)
    
    for i in range(num_bins):
        mask = bin_indices == i
        bin_counts[i] = np.sum(mask)
        if bin_counts[i] > 0:
            bin_accuracies[i] = np.mean(accuracies[mask])
            bin_confidences[i] = np.mean(confidences[mask])
            
    n_total = len(confidences)
    valid = bin_counts > 0
    ece = np.sum((bin_counts[valid] / n_total) * np.abs(bin_accuracies[valid] - bin_confidences[valid]))
    
    bin_centers = bins[:-1] + np.diff(bins) / 2
    
    return bin_confidences[valid], bin_accuracies[valid], ece, bin_counts, bin_centers

In [ ]:
import os
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


def plot_combined_calibration(results_dir: str, outfile: str = "../../outputs/figures/calibration.pdf",
                              figsize=(7.0, 3.5), square: bool = True):
    # --- seaborn theming (aesthetic only; plotting stays explicit) -----------
    sns.set_theme(context="paper", style="ticks")
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "svg.fonttype": "none",
        "pdf.fonttype": 42,   # editable text in the PDF (TrueType, not Type-3)
        "axes.linewidth": 0.8,
    })

    fig, ax = plt.subplots(figsize=figsize, dpi=300)

    # Perfect-calibration diagonal (no legend entry; obvious from the plot)
    ax.plot([0, 1], [0, 1], ls=(0, (4, 3)), lw=1.0, color="#333333", zorder=1)

    # Shade + label the overconfident region (confidence > accuracy -> below diagonal)
    ax.fill_between([0, 1], [0, 1], [0, 0], color="#d62728", alpha=0.05, zorder=0, lw=0)

    handles = []
    for filename, cfg in MODELS.items():
        filepath = os.path.join(results_dir, filename)
        if not os.path.exists(filepath):
            continue
        with open(filepath, "r") as f:
            results = json.load(f)
        if "probabilities" not in results[0]:
            continue
        bin_conf, bin_acc, ece, bin_counts, bin_centers = calculate_calibration_data(results)
        bin_conf = np.asarray(bin_conf, float)
        bin_acc = np.asarray(bin_acc, float)

        face = cfg["color"] if cfg["fill"] else "white"
        ax.plot(bin_conf, bin_acc, marker=cfg["marker"], markersize=6,
                color=cfg["color"], markerfacecolor=face,
                markeredgecolor=cfg["color"], markeredgewidth=1.3,
                linestyle=cfg["ls"], linewidth=1.6, zorder=3, alpha=0.95)

        # proxy handle for a clean legend (model name only)
        handles.append(mpl.lines.Line2D(
            [], [], color=cfg["color"], marker=cfg["marker"], markersize=6,
            markerfacecolor=face, markeredgecolor=cfg["color"], markeredgewidth=1.3,
            linestyle=cfg["ls"], linewidth=1.6, label=cfg["label"]))

    # --- formatting ----------------------------------------------------------
    lo = 0.45
    ax.set_xlim(lo, 1.0)
    ax.set_ylim(lo, 1.0)
    if square:
        ax.set_aspect("equal", adjustable="box")   # true 45deg diagonal
    ax.set_xlabel("Mean predicted confidence", fontsize=9, labelpad=6)
    ax.set_ylabel("Empirical accuracy", fontsize=9, labelpad=6)
    ax.tick_params(axis="both", which="major", labelsize=8, colors="#333333", length=3)
    sns.despine(ax=ax)   # drop top/right spines

    # "overconfident" annotation inside the shaded wedge
    ax.text(0.9, 0.5, "overconfident", fontsize=7.5, style="italic",
            color="#b03030", ha="center", va="center")

    # Frameless legend, model names only, inside the plot's upper-left whitespace
    ax.legend(handles=handles, frameon=False, fontsize=7.6,
              loc="upper left", handletextpad=0.5, labelspacing=0.5,
              borderaxespad=0.4)

    fig.tight_layout()

    # Save PDF (vector, for LaTeX) and a PNG companion
    os.makedirs(os.path.dirname(outfile) or ".", exist_ok=True)
    fig.savefig(outfile, format="pdf", bbox_inches="tight")
    fig.savefig(os.path.splitext(outfile)[0] + ".png", dpi=300, bbox_inches="tight")
    return fig

In [ ]:
def generate_latex_table(metrics_list: list) -> str:
    """Transforms the metrics dictionary into a publication-ready LaTeX table."""
    df = pd.DataFrame(metrics_list)
    
    # Identify which columns to maximize vs minimize for automatic bolding
    numeric_df = df.copy()
    for col in df.columns[1:]:
        numeric_df[col] = pd.to_numeric(numeric_df[col], errors="coerce")
        
    best_vals = {}
    for col in df.columns[1:]:
        if '↑' in col or 'Correct' in col:
            best_vals[col] = numeric_df[col].max()
        elif '↓' in col or 'Incorrect' in col:
            best_vals[col] = numeric_df[col].min()

    # Build LaTeX structural syntax
    latex = []
    latex.append("\\begin{table*}[t]")
    latex.append("\\centering")
    
    # Dynamic column formatting (@{}lccccc...@{})
    col_format = "@{}l" + "c" * (len(df.columns) - 1) + "@{}"
    latex.append(f"\\begin{{tabular}}{{{col_format}}}")
    latex.append("\\toprule")
    
    # Headers (Replacing python arrows with LaTeX math arrows)
    headers = []
    for col in df.columns:
        header_clean = col.replace(" (↑)", " ($\\uparrow$)").replace(" (↓)", " ($\\downarrow$)")
        headers.append(f"\\textbf{{{header_clean}}}")
    latex.append(" & ".join(headers) + " \\\\")
    latex.append("\\midrule")
    
    # Rows
    for _, row in df.iterrows():
        row_strs = [str(row[df.columns[0]])] # Model/Config name
        for col in df.columns[1:]:
            val = numeric_df.loc[_, col]
            if pd.isna(val):
                row_strs.append("-")
                continue
            val_str = f"{val:.4f}"
            
            # Apply \textbf{} to the best performing metrics
            if val == best_vals[col]:
                row_strs.append(f"\\textbf{{{val_str}}}")
            else:
                row_strs.append(val_str)
        latex.append(" & ".join(row_strs) + " \\\\")
        
    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\caption{\\textbf{Comparative Model Performance and Calibration.} "
                 "Results detail discrete accuracy and probabilistic calibration across all evaluated configurations. "
                 "Best values per metric are \\textbf{bolded}.}")
    latex.append("\\label{tab:calibration_metrics}")
    latex.append("\\end{table*}")
    
    return "\n".join(latex)

def evaluate_models(results_dir: str):
    """Evaluates all JSON result files and outputs a LaTeX table."""
    
    metrics_list = []
    
    for filename, cfg in MODELS.items():
        filepath = os.path.join(results_dir, filename)
        if not os.path.exists(filepath):
            continue
        
        with open(filepath, "r") as f:
            results = json.load(f)
            
        y_true = np.array([1 if str(r["true_label"]) == "1" else 0 for r in results])
        y_pred = np.array([1 if str(r["prediction"]) == "1" else 0 for r in results])
        accuracies = np.array([1 if str(r["true_label"]) == str(r["prediction"]) else 0 for r in results])

        # Proprietary baseline: logits are not exposed, so only accuracy is reported
        if "probabilities" not in results[0]:
            metrics_list.append({"Model": cfg["label"], "Accuracy (↑)": f"{np.mean(accuracies):.4f}"})
            continue
        
        prob_1 = np.array([r["probabilities"]["1"] / (r["probabilities"]["0"] + r["probabilities"]["1"]) for r in results])
        # Max confidence for calibration and conditional analysis (Also Normalized)
        confidences = np.array([
            max(r["probabilities"]["0"], r["probabilities"]["1"]) / (r["probabilities"]["0"] + r["probabilities"]["1"]) 
            for r in results
        ])

        acc_score = np.mean(accuracies)
        brier = brier_score_loss(y_true, prob_1)
        
        try:
            auc = roc_auc_score(y_true, prob_1)
        except ValueError:
            auc = np.nan
            
        ece = calculate_calibration_data(results)[2] 
        conf_correct = np.mean(confidences[accuracies == 1])
        conf_incorrect = np.mean(confidences[accuracies == 0])
        
        metrics_list.append({
            "Model": cfg["label"],
            "Accuracy (↑)": f"{acc_score:.4f}",
            "ROC-AUC (↑)": f"{auc:.4f}",
            "Brier Score (↓)": f"{brier:.4f}",
            "ECE (↓)": f"{ece:.4f}",
            "Conf Correct (↑)": f"{conf_correct:.4f}",
            "Conf Incorrect (↓)": f"{conf_incorrect:.4f}"
        })

    # Print the ready-to-copy LaTeX code
    print(generate_latex_table(metrics_list))

In [ ]:
plot_combined_calibration("../../outputs/results/")

evaluate_models("../../outputs/results/")